In [19]:
import pandas as pd

# Load datasets once
ehr = pd.read_csv("Ops Case Study Dataset - Sample EHR Data.csv")
db  = pd.read_csv("Ops Case Study Dataset - Sample DB Data.csv")

# Normalize column names
ehr.columns = ehr.columns.str.strip().str.lower().str.replace(" ", "_")
db.columns  = db.columns.str.strip().str.lower().str.replace(" ", "_")

print("EHR columns:", ehr.columns.tolist())
print("DB columns:", db.columns.tolist())

EHR columns: ['patient_name', 'provider_name', 'date_of_service', 'cpt_code']
DB columns: ['patient_name', 'provider_name', 'from_date_range', 'cpt_codes']


In [21]:
def normalize(s):
    return s.astype(str).str.strip().str.lower()

In [23]:
# EHR date
ehr["date_of_service"] = pd.to_datetime(ehr["date_of_service"], errors="coerce")

# DB date range → extract start date
db["service_date"] = (
    db["from_date_range"].astype(str)
    .str.extract(r'(\d{2}/\d{2}/\d{4}|\d{4}-\d{2}-\d{2})')[0]
)
db["service_date"] = pd.to_datetime(db["service_date"], errors="coerce")

In [25]:
ehr["Encounter_ID"] = (
    normalize(ehr["patient_name"]) + "_" +
    ehr["date_of_service"].dt.strftime("%Y-%m-%d") + "_" +
    normalize(ehr["provider_name"])
)

db["Encounter_ID"] = (
    normalize(db["patient_name"]) + "_" +
    db["service_date"].dt.strftime("%Y-%m-%d") + "_" +
    normalize(db["provider_name"])
)

In [27]:
ehr_encounters = ehr.drop_duplicates(subset=["Encounter_ID"])
db_encounters  = db.drop_duplicates(subset=["Encounter_ID"])

In [29]:
missing = ehr_encounters[~ehr_encounters["Encounter_ID"].isin(db_encounters["Encounter_ID"])].copy()

deliverable1 = missing[["patient_name","provider_name","date_of_service","cpt_code","Encounter_ID"]]
print("Missing encounters:", len(deliverable1))
deliverable1.head()

Missing encounters: 283


,patient_name,provider_name,date_of_service,cpt_code,Encounter_ID
22,Olivia Miller,Sebastian Miller,2024-07-23,97140,olivia miller_2024-07-23_sebastian miller
30,Noah Gonzalez,Elijah Johnson,2024-08-01,97140,noah gonzalez_2024-08-01_elijah johnson
72,Julian King,Noah Lee,2024-09-11,97112,julian king_2024-09-11_noah lee
76,Layla Miller,Aiden King,2024-08-14,97140,layla miller_2024-08-14_aiden king
83,Sebastian Jackson,Sebastian Miller,2024-09-26,TOS115,sebastian jackson_2024-09-26_sebastian miller


In [31]:
missing = ehr_encounters[~ehr_encounters["Encounter_ID"].isin(db_encounters["Encounter_ID"])].copy()

deliverable1 = missing[["patient_name","provider_name","date_of_service","cpt_code","Encounter_ID"]]
print("Missing encounters:", len(deliverable1))
deliverable1.head()

Missing encounters: 283


,patient_name,provider_name,date_of_service,cpt_code,Encounter_ID
22,Olivia Miller,Sebastian Miller,2024-07-23,97140,olivia miller_2024-07-23_sebastian miller
30,Noah Gonzalez,Elijah Johnson,2024-08-01,97140,noah gonzalez_2024-08-01_elijah johnson
72,Julian King,Noah Lee,2024-09-11,97112,julian king_2024-09-11_noah lee
76,Layla Miller,Aiden King,2024-08-14,97140,layla miller_2024-08-14_aiden king
83,Sebastian Jackson,Sebastian Miller,2024-09-26,TOS115,sebastian jackson_2024-09-26_sebastian miller


In [33]:
by_provider = missing.groupby("provider_name").size().reset_index(name="Missing Count")
by_date     = missing.groupby("date_of_service").size().reset_index(name="Missing Count")
by_cpt      = missing.groupby("cpt_code").size().reset_index(name="Missing Count")

print(by_provider.head())
print(by_date.head())
print(by_cpt.head())

        provider_name  Missing Count
0          Aiden King             95
1  Charlotte Williams             20
2      Elijah Johnson             10
3          Julian Lee             13
4          Liam Young             36
  date_of_service  Missing Count
0      2024-07-01              2
1      2024-07-02              1
2      2024-07-03              2
3      2024-07-05              2
4      2024-07-08              5
  cpt_code  Missing Count
0    97010             31
1    97012              1
2    97014             15
3    97032              1
4    97035              2


In [37]:
with pd.ExcelWriter("Closed_Encounters_Analysis.xlsx", engine="openpyxl") as w:
    deliverable1.to_excel(w, sheet_name="Missing Encounters", index=False)
    by_provider.to_excel(w, sheet_name="By Provider", index=False)
    by_date.to_excel(w, sheet_name="By Date", index=False)
    by_cpt.to_excel(w, sheet_name="By CPT", index=False)